# Knowing–Testing Gap: first pipeline run (Colab)

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

This notebook runs stages 1–4 on 10 HumanEval+ problems with Qwen2.5-Coder-0.5B-Instruct:
1. the model writes 5 solutions per problem
2. EvalPlus labels each solution correct or buggy
3. the model writes unit tests for each of its own solutions
4. we run those tests and count how often they miss the model's own bugs

Run the cells from top to bottom. The whole run takes roughly 10–20 minutes on a T4.

In [ ]:
!nvidia-smi
!pip install -q evalplus==0.3.1

## Save results to Google Drive
Colab deletes files when the session ends, so results are written to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS = '/content/drive/MyDrive/ktgap_results'
os.makedirs(RESULTS, exist_ok=True)
!mkdir -p /content/ktgap/src
%cd /content/ktgap

## Write the code files
(Same files as in the GitHub repo.)

In [ ]:
%%writefile src/common.py
"""Shared helpers for the knowing-testing gap pipeline."""
import ast
import json
import os
import re
import subprocess
import sys

# ---------------------------------------------------------------- datasets

def load_problems(dataset):
    """Return {task_id: problem} for 'humaneval' or 'mbpp' (EvalPlus versions)."""
    from evalplus.data import get_human_eval_plus, get_mbpp_plus
    if dataset == "humaneval":
        return get_human_eval_plus()
    if dataset == "mbpp":
        return get_mbpp_plus()
    raise ValueError(f"unknown dataset: {dataset}")


def canonical_code(dataset, problem):
    """Full reference implementation (used to check whether a test is valid)."""
    if dataset == "humaneval":
        return problem["prompt"] + problem["canonical_solution"]
    return problem["canonical_solution"]  # MBPP+ stores the full function


def select_tasks(problems, limit=None, task_ids=None):
    ids = list(problems)
    if task_ids:
        ids = [t for t in ids if t in set(task_ids)]
    if limit:
        ids = ids[:limit]
    return ids

# ---------------------------------------------------------------- jsonl io

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

# ---------------------------------------------------------------- code extraction

_FENCE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.DOTALL)


def extract_code(text):
    """Return the longest fenced code block, or the raw text if there is none."""
    blocks = _FENCE.findall(text)
    if not blocks:
        # model may have stopped before closing the fence
        m = re.search(r"```(?:python|py)?\s*\n(.*)", text, re.DOTALL)
        return (m.group(1) if m else text).strip()
    return max(blocks, key=len).strip()


def extract_asserts(text, max_asserts=None):
    """Pull assert statements out of a model response (handles multi-line asserts)."""
    code = extract_code(text)
    asserts = []
    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.Assert):
                seg = ast.get_source_segment(code, node)
                if seg:
                    asserts.append(seg)
    except SyntaxError:
        # fall back to single-line asserts that parse on their own
        for line in code.splitlines():
            line = line.strip()
            if line.startswith("assert"):
                try:
                    ast.parse(line)
                    asserts.append(line)
                except SyntaxError:
                    pass
    seen, unique = set(), []
    for a in asserts:  # drop duplicates, keep order
        if a not in seen:
            seen.add(a)
            unique.append(a)
    return unique[:max_asserts] if max_asserts else unique

# ---------------------------------------------------------------- sandboxed execution

_RESULT_MARK = "__KTGAP_RESULT__"

_HARNESS = r'''
import json, signal, sys
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (2 * 1024**3, 2 * 1024**3))
except Exception:
    pass
payload = json.loads(sys.stdin.read())
T = payload["per_assert_timeout"]

class _Timeout(BaseException):
    pass

def _handler(signum, frame):
    raise _Timeout()

signal.signal(signal.SIGALRM, _handler)
ns = {"__name__": "__ktgap__"}
load = "ok"
try:
    signal.alarm(T)
    exec(payload["code"], ns)
    signal.alarm(0)
except BaseException as e:
    signal.alarm(0)
    load = "error:" + type(e).__name__
results = []
for a in payload["asserts"]:
    if load != "ok":
        results.append("not_run")
        continue
    try:
        signal.alarm(T)
        exec(a, ns)
        signal.alarm(0)
        results.append("pass")
    except AssertionError:
        signal.alarm(0)
        results.append("fail")
    except _Timeout:
        results.append("timeout")
    except BaseException as e:
        signal.alarm(0)
        results.append("error:" + type(e).__name__)
sys.stdout.write("\n''' + _RESULT_MARK + r'''" + json.dumps({"load": load, "results": results}) + "\n")
'''


def run_asserts(code, asserts, per_assert_timeout=3, total_timeout=60):
    """Run each assert against `code` in a separate Python process.

    Returns {"load": "ok" | "error:...", "results": ["pass"|"fail"|"timeout"|"error:X"|...]}.
    Untrusted model code runs here: use only on a VM / Colab, never on a machine you care about.
    """
    if not asserts:
        return {"load": "ok", "results": []}
    payload = json.dumps({"code": code, "asserts": asserts,
                          "per_assert_timeout": per_assert_timeout})
    try:
        proc = subprocess.run([sys.executable, "-c", _HARNESS], input=payload,
                              capture_output=True, text=True, timeout=total_timeout)
        for line in proc.stdout.splitlines()[::-1]:
            if line.startswith(_RESULT_MARK):
                return json.loads(line[len(_RESULT_MARK):])
        return {"load": "crash", "results": ["crash"] * len(asserts)}
    except subprocess.TimeoutExpired:
        return {"load": "timeout", "results": ["timeout"] * len(asserts)}


In [ ]:
%%writefile src/generate.py
"""Stage 1 and 3 of the pipeline: generate solutions, then generate unit tests.

Examples
--------
# 1) sample 5 solutions per problem for 10 HumanEval+ problems
python src/generate.py solutions --model Qwen/Qwen2.5-Coder-0.5B-Instruct \
    --dataset humaneval --limit 10 --n 5 --out results/qwen0.5b/solutions.jsonl

# 3) have the same model write tests for each of its own solutions (white-box)
python src/generate.py tests --model Qwen/Qwen2.5-Coder-0.5B-Instruct \
    --dataset humaneval --solutions results/qwen0.5b/solutions_labeled.jsonl \
    --mode code --out results/qwen0.5b/selftests.jsonl
"""
import argparse
import time

from common import extract_asserts, extract_code, load_problems, read_jsonl, select_tasks, write_jsonl

SYSTEM = "You are a careful Python programmer."

SOLUTION_PROMPT = """Complete the following Python task. The function must be named `{entry}`.
Respond with the complete function (including imports and the signature) in a single ```python code block.

```python
{prompt}
```"""

TEST_PROMPT_CODE = """Below is a programming task and an implementation of it.

Task:
```python
{prompt}
```

Implementation:
```python
{solution}
```

Write {k} unit tests for `{entry}` as Python assert statements. Each test must check that a specific input produces the output the task description requires. Respond with a single ```python code block that contains only the assert statements."""

TEST_PROMPT_SPEC = """Below is a programming task.

Task:
```python
{prompt}
```

Write {k} unit tests for `{entry}` as Python assert statements. Each test must check that a specific input produces the output the task description requires. Respond with a single ```python code block that contains only the assert statements."""


# ---------------------------------------------------------------- backends

class HFBackend:
    """Hugging Face transformers. Works on CPU and on a single GPU (Colab, Jetstream2)."""

    def __init__(self, model_name, seed, batch_size):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch = torch
        torch.manual_seed(seed)
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        if self.tok.chat_template is None:
            raise SystemExit("This model has no chat template; use an *-Instruct model.")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else (
            torch.float16 if device == "cuda" else torch.float32)
        # full precision weights (no quantization): hidden states must match the real model
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(device)
        self.model.eval()
        self.device = device
        self.batch_size = batch_size

    def chat_text(self, user_msg):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def generate(self, prompts, n, temperature, top_p, max_new_tokens):
        outs = []
        for i in range(0, len(prompts), self.batch_size):
            batch = prompts[i:i + self.batch_size]
            enc = self.tok(batch, return_tensors="pt", padding=True).to(self.device)
            kwargs = dict(max_new_tokens=max_new_tokens, num_return_sequences=n,
                          pad_token_id=self.tok.pad_token_id)
            if temperature > 0:
                kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
            else:
                kwargs.update(do_sample=False)
            with self.torch.no_grad():
                gen = self.model.generate(**enc, **kwargs)
            gen = gen[:, enc["input_ids"].shape[1]:]
            texts = self.tok.batch_decode(gen, skip_special_tokens=True)
            for j in range(len(batch)):
                outs.append(texts[j * n:(j + 1) * n])
            print(f"  generated {min(i + self.batch_size, len(prompts))}/{len(prompts)} prompts", flush=True)
        return outs


class VLLMBackend:
    """vLLM: much faster on a GPU (use on Jetstream2). Does not return hidden states."""

    def __init__(self, model_name, seed, batch_size):
        from vllm import LLM
        self.llm = LLM(model=model_name, dtype="bfloat16", seed=seed)
        self.tok = self.llm.get_tokenizer()
        self.seed = seed

    def chat_text(self, user_msg):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def generate(self, prompts, n, temperature, top_p, max_new_tokens):
        from vllm import SamplingParams
        params = SamplingParams(n=n, temperature=temperature, top_p=top_p if temperature > 0 else 1.0,
                                max_tokens=max_new_tokens, seed=self.seed)
        results = self.llm.generate(prompts, params)
        return [[o.text for o in r.outputs] for r in results]


def make_backend(args):
    cls = VLLMBackend if args.backend == "vllm" else HFBackend
    return cls(args.model, args.seed, args.batch_size)

# ---------------------------------------------------------------- stages

def run_solutions(args):
    problems = load_problems(args.dataset)
    task_ids = select_tasks(problems, args.limit, args.task_ids)
    backend = make_backend(args)
    prompts = [backend.chat_text(SOLUTION_PROMPT.format(prompt=problems[t]["prompt"].strip(),
                                                        entry=problems[t]["entry_point"]))
               for t in task_ids]
    t0 = time.time()
    outs = backend.generate(prompts, args.n, args.temperature, args.top_p, args.max_new_tokens)
    rows = []
    for t, prompt_text, samples in zip(task_ids, prompts, outs):
        for k, raw in enumerate(samples):
            rows.append({"task_id": t, "sample_id": k, "dataset": args.dataset,
                         "solution_model": args.model, "prompt_text": prompt_text,
                         "raw": raw, "solution": extract_code(raw),
                         "temperature": args.temperature, "seed": args.seed})
    write_jsonl(args.out, rows)
    print(f"wrote {len(rows)} solutions to {args.out} in {time.time() - t0:.0f}s")


def run_tests(args):
    problems = load_problems(args.dataset)
    backend = make_backend(args)
    if args.mode == "code":
        targets = read_jsonl(args.solutions)  # one test set per solution (white-box)
        if args.only_buggy_and_n_correct is not None:
            targets = _subsample(targets, args.only_buggy_and_n_correct)
        prompts = [backend.chat_text(TEST_PROMPT_CODE.format(
            prompt=problems[r["task_id"]]["prompt"].strip(), solution=r["solution"],
            entry=problems[r["task_id"]]["entry_point"], k=args.k)) for r in targets]
    else:  # spec: one test set per task (black-box), shared by all solutions of that task
        sols = read_jsonl(args.solutions)
        task_ids = sorted({r["task_id"] for r in sols}, key=list(problems).index)
        targets = [{"task_id": t, "sample_id": None, "solution_model": None} for t in task_ids]
        prompts = [backend.chat_text(TEST_PROMPT_SPEC.format(
            prompt=problems[t]["prompt"].strip(), entry=problems[t]["entry_point"], k=args.k))
            for t in task_ids]
    t0 = time.time()
    outs = backend.generate(prompts, 1, args.temperature, args.top_p, args.max_new_tokens)
    rows = []
    for r, prompt_text, samples in zip(targets, prompts, outs):
        raw = samples[0]
        rows.append({"task_id": r["task_id"], "sample_id": r["sample_id"], "dataset": args.dataset,
                     "solution_model": r.get("solution_model"), "test_model": args.model,
                     "mode": args.mode, "prompt_text": prompt_text, "raw": raw,
                     "asserts": extract_asserts(raw, max_asserts=args.k)})
    write_jsonl(args.out, rows)
    empty = sum(1 for r in rows if not r["asserts"])
    print(f"wrote {len(rows)} test sets to {args.out} in {time.time() - t0:.0f}s "
          f"({empty} produced no parsable asserts)")


def _subsample(rows, n_correct):
    """Keep every buggy solution plus up to n_correct correct ones per task (needs labels)."""
    if "correct" not in rows[0]:
        raise SystemExit("--only-buggy-and-n-correct needs a labeled solutions file (run evaluate.py label)")
    kept, seen = [], {}
    for r in rows:
        if not r["correct"]:
            kept.append(r)
        elif seen.get(r["task_id"], 0) < n_correct:
            kept.append(r)
            seen[r["task_id"]] = seen.get(r["task_id"], 0) + 1
    return kept


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["solutions", "tests"])
    ap.add_argument("--model", required=True)
    ap.add_argument("--dataset", choices=["humaneval", "mbpp"], default="humaneval")
    ap.add_argument("--backend", choices=["hf", "vllm"], default="hf")
    ap.add_argument("--out", required=True)
    ap.add_argument("--limit", type=int, default=None, help="first N problems only")
    ap.add_argument("--task-ids", nargs="*", default=None)
    ap.add_argument("--n", type=int, default=5, help="solutions per problem")
    ap.add_argument("--k", type=int, default=5, help="asserts requested per test set")
    ap.add_argument("--solutions", help="(tests stage) solutions file, labeled or not")
    ap.add_argument("--mode", choices=["code", "spec"], default="code",
                    help="code = tester sees the solution (own-code tests); spec = docstring only")
    ap.add_argument("--only-buggy-and-n-correct", type=int, default=None,
                    help="(tests stage) keep all buggy solutions + N correct ones per task")
    ap.add_argument("--temperature", type=float, default=None)
    ap.add_argument("--top-p", type=float, default=0.95)
    ap.add_argument("--max-new-tokens", type=int, default=512)
    ap.add_argument("--batch-size", type=int, default=4)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    if args.temperature is None:  # diverse solutions (we need bugs); near-deterministic tests
        args.temperature = 0.8 if args.stage == "solutions" else 0.2
    if args.stage == "solutions":
        run_solutions(args)
    else:
        if not args.solutions:
            raise SystemExit("tests stage needs --solutions")
        run_tests(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""Stage 2 and 4 of the pipeline: label solutions, then run the generated tests.

Examples
--------
# 2) label each solution correct/buggy with the EvalPlus hidden tests
python src/evaluate.py label --dataset humaneval \
    --solutions results/qwen0.5b/solutions.jsonl --out results/qwen0.5b/solutions_labeled.jsonl

# 4) run each test set against its solution; report how often own tests miss own bugs
python src/evaluate.py run-tests --dataset humaneval \
    --solutions results/qwen0.5b/solutions_labeled.jsonl \
    --tests results/qwen0.5b/selftests.jsonl --out results/qwen0.5b/selftest_results.jsonl
"""
import argparse
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

from common import canonical_code, load_problems, read_jsonl, run_asserts, write_jsonl

# ---------------------------------------------------------------- label (EvalPlus)

def _expected_outputs(dataset, problems):
    from evalplus.data import get_human_eval_plus_hash, get_mbpp_plus_hash
    from evalplus.evaluate import MBPP_OUTPUT_NOT_NONE_TASKS, get_groundtruth
    if dataset == "humaneval":
        return get_groundtruth(problems, get_human_eval_plus_hash(), [])
    return get_groundtruth(problems, get_mbpp_plus_hash(), MBPP_OUTPUT_NOT_NONE_TASKS)


def _check(args):
    from evalplus.evaluate import check_correctness
    dataset, i, problem, solution, expected = args
    r = check_correctness(dataset, i, problem, solution, expected, False, True, i)
    return i, r["base"][0], r["plus"][0]


def run_label(args):
    problems = load_problems(args.dataset)
    expected = _expected_outputs(args.dataset, problems)  # cached by EvalPlus after first run
    rows = read_jsonl(args.solutions)
    jobs = [(args.dataset, i, problems[r["task_id"]], r["solution"], expected[r["task_id"]])
            for i, r in enumerate(rows)]
    with ProcessPoolExecutor(max_workers=args.workers) as ex:
        for i, base, plus in ex.map(_check, jobs):
            rows[i]["base_status"] = base
            rows[i]["plus_status"] = plus
            rows[i]["correct"] = base == "pass" and plus == "pass"
    write_jsonl(args.out, rows)
    n_ok = sum(r["correct"] for r in rows)
    n_base_only = sum(r["base_status"] == "pass" and not r["correct"] for r in rows)
    print(f"labeled {len(rows)} solutions: {n_ok} correct, {len(rows) - n_ok} buggy "
          f"({n_base_only} buggy ones pass the original tests but fail EvalPlus extra tests)")

# ---------------------------------------------------------------- run-tests

def run_tests(args):
    problems = load_problems(args.dataset)
    sols = read_jsonl(args.solutions)
    if "correct" not in sols[0]:
        raise SystemExit("run-tests needs a labeled solutions file (run the label stage first)")
    tests = read_jsonl(args.tests)
    by_id = {(r["task_id"], r["sample_id"]): r for r in sols}

    # pair every test set with the solution(s) it should be run against
    pairs = []
    for t in tests:
        if t["mode"] == "spec":  # one test set per task -> every solution of that task
            pairs += [(t, s) for s in sols if s["task_id"] == t["task_id"]]
        elif (t["task_id"], t["sample_id"]) in by_id:
            pairs.append((t, by_id[(t["task_id"], t["sample_id"])]))

    # a test is "valid" if it passes on the reference solution (otherwise its expected value is wrong)
    def validity(task_id, asserts):
        code = canonical_code(args.dataset, problems[task_id])
        res = run_asserts(code, asserts, args.per_assert_timeout)
        return [x == "pass" for x in res["results"]]

    def on_solution(pair):
        t, s = pair
        return run_asserts(s["solution"], t["asserts"], args.per_assert_timeout)

    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        valid_list = list(ex.map(lambda t: validity(t["task_id"], t["asserts"]), tests))
        sol_results = list(ex.map(on_solution, pairs))
    valid_by_test = {id(t): v for t, v in zip(tests, valid_list)}

    out = []
    for (t, s), res in zip(pairs, sol_results):
        valid = valid_by_test[id(t)]
        passed = [x == "pass" for x in res["results"]]
        n = len(passed)
        all_pass = n > 0 and all(passed)  # what a developer would see: "tests are green"
        valid_catch = any(v and not p for v, p in zip(valid, passed))
        out.append({
            "task_id": s["task_id"], "sample_id": s["sample_id"],
            "solution_model": s["solution_model"], "test_model": t["test_model"],
            "mode": t["mode"], "self_test": t["test_model"] == s["solution_model"],
            "correct": s["correct"], "n_asserts": n, "n_valid": sum(valid),
            "assert_results": res["results"], "assert_valid": valid,
            "tests_all_pass": all_pass,
            "valid_test_catches_bug": (not s["correct"]) and valid_catch,
            "missed_bug": (not s["correct"]) and all_pass,     # key outcome for RQ1/RQ2
            "false_alarm": s["correct"] and n > 0 and not all_pass,
        })
    write_jsonl(args.out, out)
    summarize(out)


def summarize(rows):
    def pct(a, b):
        return f"{a}/{b} ({100 * a / b:.1f}%)" if b else f"{a}/0"
    usable = [r for r in rows if r["n_asserts"] > 0]
    buggy = [r for r in usable if not r["correct"]]
    correct = [r for r in usable if r["correct"]]
    n_asserts = sum(r["n_asserts"] for r in usable)
    print("\n=== summary ===")
    print(f"pairs with at least one parsable assert: {pct(len(usable), len(rows))}")
    print(f"assert validity (passes on reference solution): {pct(sum(r['n_valid'] for r in usable), n_asserts)}")
    print(f"buggy solutions whose tests are all green (MISSED BUGS): "
          f"{pct(sum(r['missed_bug'] for r in buggy), len(buggy))}")
    print(f"buggy solutions caught by at least one valid assert: "
          f"{pct(sum(r['valid_test_catches_bug'] for r in buggy), len(buggy))}")
    print(f"correct solutions with a failing test (false alarms): "
          f"{pct(sum(r['false_alarm'] for r in correct), len(correct))}")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["label", "run-tests"])
    ap.add_argument("--dataset", choices=["humaneval", "mbpp"], default="humaneval")
    ap.add_argument("--solutions", required=True)
    ap.add_argument("--tests", help="(run-tests) test sets from generate.py tests")
    ap.add_argument("--out", required=True)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--per-assert-timeout", type=int, default=3)
    args = ap.parse_args()
    if args.stage == "label":
        run_label(args)
    else:
        if not args.tests:
            raise SystemExit("run-tests needs --tests")
        run_tests(args)


if __name__ == "__main__":
    main()


## Settings

In [ ]:
MODEL = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
DATASET = 'humaneval'
LIMIT = 10   # number of problems
N = 5        # solutions per problem
D = f'{RESULTS}/qwen0.5b_{DATASET}_{LIMIT}'
os.makedirs(D, exist_ok=True)
print(D)

## Stage 1 — generate solutions

In [ ]:
!python src/generate.py solutions --model $MODEL --dataset $DATASET --limit $LIMIT --n $N --batch-size 4 --out $D/solutions.jsonl

## Stage 2 — label correct / buggy with EvalPlus
The first run computes reference outputs for all problems (about 15–60 s); later runs reuse a cache.

In [ ]:
!python src/evaluate.py label --dataset $DATASET --solutions $D/solutions.jsonl --out $D/solutions_labeled.jsonl

## Stage 3 — the model writes tests for its own solutions (white-box)

In [ ]:
!python src/generate.py tests --model $MODEL --dataset $DATASET --solutions $D/solutions_labeled.jsonl --mode code --batch-size 4 --out $D/selftests.jsonl

## Stage 4 — run the tests: how often do they miss the model's own bugs?

In [ ]:
!python src/evaluate.py run-tests --dataset $DATASET --solutions $D/solutions_labeled.jsonl --tests $D/selftests.jsonl --out $D/selftest_results.jsonl

## Look at a missed bug
A buggy solution whose own tests are all green — the cases the project is about.

In [ ]:
import json
sols = {(r['task_id'], r['sample_id']): r for r in map(json.loads, open(f'{D}/solutions_labeled.jsonl'))}
tests = {(r['task_id'], r['sample_id']): r for r in map(json.loads, open(f'{D}/selftests.jsonl'))}
missed = [r for r in map(json.loads, open(f'{D}/selftest_results.jsonl')) if r['missed_bug']]
print(len(missed), 'missed bugs')
for r in missed[:2]:
    key = (r['task_id'], r['sample_id'])
    print('=' * 70, '\n', key, '| base:', sols[key]['base_status'], '| plus:', sols[key]['plus_status'])
    print(sols[key]['solution'])
    print('--- its own tests (all pass):')
    print('\n'.join(tests[key]['asserts']))

## Done
Results are in Google Drive under `ktgap_results/`. Next steps: hidden-state extraction (`extract.py`) and probes.